# 6. Step-Level Analysis: Watching Repression Emerge

OLMo Think-SFT releases 43 training checkpoints. We trace word probabilities across 10 evenly-spaced steps to watch repression and displacement emerge during training.

In [ ]:
import pandas as pd
import numpy as np
from plotnine import *

theme_malign = theme_minimal() + theme(
    figure_size=(10, 6),
    plot_background=element_rect(fill='white'),
    panel_grid_minor=element_blank(),
    text=element_text(family='serif'),
    plot_title=element_text(size=14, weight='bold'),
    plot_subtitle=element_text(size=11, color='#666'),
)

words = pd.read_csv('data/step_analysis_words.csv')
metrics = pd.read_csv('data/step_analysis_metrics.csv')

words['category'] = words['label'].str.replace(r'_\d+$', '', regex=True)
print(f"Words: {words.shape}, steps: {sorted(words['step'].unique())}")
print(f"Tracked words: {sorted(words['word'].unique())[:20]}")

## Repression onset: sexual vs violent content

Sexual repression is a phase transition (70% drop by step 1000). Violence repression is non-monotonic.

In [ ]:
# Track key repressed words across steps
repressed = words[words.word.isin(['kill', 'fuck', 'cock'])].copy()
repressed = repressed.groupby(['step', 'word'])['probability'].mean().reset_index()

(ggplot(repressed, aes(x='step', y='probability', color='word'))
 + geom_line(size=1.2)
 + geom_point(size=2)
 + scale_color_manual(values={'kill': '#e15759', 'fuck': '#b07aa1', 'cock': '#f28e2b'})
 + labs(title='Repression onset during SFT training',
        subtitle='Sexual repression is immediate (phase transition). Violence is non-monotonic.',
        x='Training step', y='Mean probability', color='')
 + theme_malign
)

## Displacement lag: repressed words fall before displacement targets rise

Evidence of genuine emergent displacement — not simultaneous substitution.

In [ ]:
# Repression + displacement pairs
pairs = words[words.word.isin(['kill', 'scream', 'fuck', 'kiss'])].copy()
pairs = pairs.groupby(['step', 'word'])['probability'].mean().reset_index()

# Normalise to base (step 0) for comparison
for w in pairs.word.unique():
    base_val = pairs.loc[(pairs.word == w) & (pairs.step == pairs.step.min()), 'probability'].values
    if len(base_val) > 0 and base_val[0] > 0:
        pairs.loc[pairs.word == w, 'normalised'] = pairs.loc[pairs.word == w, 'probability'] / base_val[0]

pairs['role'] = pairs['word'].map({
    'kill': 'repressed', 'fuck': 'repressed',
    'scream': 'displaced to', 'kiss': 'displaced to'
})

(ggplot(pairs, aes(x='step', y='normalised', color='word', linetype='role'))
 + geom_line(size=1.2)
 + geom_point(size=2)
 + geom_hline(yintercept=1.0, linetype='dotted', color='#666')
 + scale_color_manual(values={'kill': '#e15759', 'scream': '#4e79a7', 'fuck': '#b07aa1', 'kiss': '#59a14f'})
 + labs(title='Displacement lag: repression precedes displacement',
        subtitle='Normalised to base probability. Repressed words fall first; displacement targets rise later.',
        x='Training step', y='Probability relative to base', color='', linetype='')
 + theme_malign
 + theme(figure_size=(12, 6))
)